---
title: "Acronyms"
format:
  html:
    code-fold: true
    code-summary: "Show code"
---

This notebook is available to use, reproduce, and validate the acronym results for ClimateKG.
Source notebook: [acronyms.ipynb](../../sparql_queries/acronyms.ipynb)

It checks whether acronym entries are currently represented in the live ClimateKG instance and how they are linked to reports.

We validate the live graph before interpreting any results, following the same query-first pattern as the other notebooks in this series.

**Query UI**: https://climatekg.tibwiki.io/query/

**SPARQL endpoint**: https://climatekg.tibwiki.io/query/proxy/sparql

In [1]:
import os
import pandas as pd
from SPARQLWrapper import JSON
from IPython.display import display, Markdown

from wikibase_auth import DEFAULT_SPARQL_ENDPOINT, DEFAULT_WIKIBASE_URL, build_sparql_client

os.environ.setdefault("CLIMATEKG_SPARQL_USERNAME", "ckg")
os.environ.setdefault("CLIMATEKG_SPARQL_PASSWORD", "fairdata")

SPARQL_ENDPOINT = os.getenv("CLIMATEKG_SPARQL_ENDPOINT", DEFAULT_SPARQL_ENDPOINT)
WIKIBASE_URL = os.getenv("CLIMATEKG_WIKIBASE_URL", DEFAULT_WIKIBASE_URL)

print(f"Endpoint: {SPARQL_ENDPOINT}")
print("Libraries imported successfully")

'ckg'

'fairdata'

Endpoint: https://climatekg.tibwiki.io/query/proxy/sparql
Libraries imported successfully


In [2]:
from pathlib import Path

entity_ns = f"{WIKIBASE_URL.rstrip('/')}/entity/"
property_ns = f"{WIKIBASE_URL.rstrip('/')}/prop/direct/"

count_query = f'''
PREFIX wd: <{entity_ns}>
PREFIX wdt: <{property_ns}>

SELECT (COUNT(DISTINCT ?acronym) AS ?acronymCount)
WHERE {{
  ?acronym wdt:P1 wd:Q2087 .
}}
'''

display(Markdown("**SPARQL query (copy/paste into ClimateKG Query UI: https://climatekg.tibwiki.io/query/)**"))
display(Markdown("```sparql\n" + count_query + "\n```"))

sparql = build_sparql_client(SPARQL_ENDPOINT)
sparql.setQuery(count_query)
sparql.setReturnFormat(JSON)
count_results = sparql.query().convert()
acronym_count = int(count_results["results"]["bindings"][0]["acronymCount"]["value"])

print(f"Acronym count (Q2087 distinct items): {acronym_count}")

query = f'''
PREFIX wd: <{entity_ns}>
PREFIX wdt: <{property_ns}>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?acronym ?acronymLabel ?report ?reportLabel
WHERE {{
  ?acronym wdt:P1 wd:Q2087 .
  OPTIONAL {{ ?acronym wdt:P3 ?report . }}
  OPTIONAL {{ ?acronym rdfs:label ?acronymLabel . FILTER(LANG(?acronymLabel) = "en") }}
  OPTIONAL {{ ?report rdfs:label ?reportLabel . FILTER(LANG(?reportLabel) = "en") }}
}}
ORDER BY ?acronymLabel ?reportLabel
'''

display(Markdown("**SPARQL query (copy/paste into ClimateKG Query UI: https://climatekg.tibwiki.io/query/)**"))
display(Markdown("```sparql\n" + query + "\n```"))

sparql.setQuery(query)
results = sparql.query().convert()
rows = results.get("results", {}).get("bindings", [])
df_all_acronyms = pd.DataFrame([{k: v["value"] for k, v in row.items()} for row in rows])

if not df_all_acronyms.empty:
    df_all_acronyms = df_all_acronyms.sort_values(["acronymLabel", "reportLabel"], na_position="last").reset_index(drop=True)

cwd = Path.cwd()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "docs").exists() and (p / "sparql_queries").exists()), cwd)
csv_path = repo_root / "docs" / "sparql_queries" / "acronyms.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)
df_all_acronyms.to_csv(csv_path, index=False)

print(f"Rows returned in acronym table: {len(df_all_acronyms)}")
print(f"CSV saved to: {csv_path}")
display(Markdown("Download CSV: [acronyms.csv](acronyms.csv)"))
display(df_all_acronyms.head(50))

**SPARQL query (copy/paste into ClimateKG Query UI: https://climatekg.tibwiki.io/query/)**

```sparql

PREFIX wd: <https://climatekg.tibwiki.io/entity/>
PREFIX wdt: <https://climatekg.tibwiki.io/prop/direct/>

SELECT (COUNT(DISTINCT ?acronym) AS ?acronymCount)
WHERE {
  ?acronym wdt:P1 wd:Q2087 .
}

```

Acronym count (Q2087 distinct items): 1910


**SPARQL query (copy/paste into ClimateKG Query UI: https://climatekg.tibwiki.io/query/)**

```sparql

PREFIX wd: <https://climatekg.tibwiki.io/entity/>
PREFIX wdt: <https://climatekg.tibwiki.io/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?acronym ?acronymLabel ?report ?reportLabel
WHERE {
  ?acronym wdt:P1 wd:Q2087 .
  OPTIONAL { ?acronym wdt:P3 ?report . }
  OPTIONAL { ?acronym rdfs:label ?acronymLabel . FILTER(LANG(?acronymLabel) = "en") }
  OPTIONAL { ?report rdfs:label ?reportLabel . FILTER(LANG(?reportLabel) = "en") }
}
ORDER BY ?acronymLabel ?reportLabel

```

Rows returned in acronym table: 2993
CSV saved to: C:\git\Climate-KG-data\docs\sparql_queries\acronyms.csv


Download CSV: [acronyms.csv](acronyms.csv)

,report,reportLabel,acronym,acronymLabel
0,https://climatekg.tibwiki.io/entity/Q10,Special Report: Global Warming of 1.5°C,https://climatekg.tibwiki.io/entity/Q2089,1.5DS
1,https://climatekg.tibwiki.io/entity/Q77,Working Group I: Climate Change 2021 – The Phy...,https://climatekg.tibwiki.io/entity/Q3038,20CR
2,https://climatekg.tibwiki.io/entity/Q10,Special Report: Global Warming of 1.5°C,https://climatekg.tibwiki.io/entity/Q2090,2DS
3,https://climatekg.tibwiki.io/entity/Q35,Special Report: Climate Change and Land,https://climatekg.tibwiki.io/entity/Q2649,3-NOP
4,https://climatekg.tibwiki.io/entity/Q106,Working Group II: Climate Change 2022 – Impact...,https://climatekg.tibwiki.io/entity/Q3325,3-PG
5,https://climatekg.tibwiki.io/entity/Q35,Special Report: Climate Change and Land,https://climatekg.tibwiki.io/entity/Q2650,A/F
6,https://climatekg.tibwiki.io/entity/Q106,Working Group II: Climate Change 2022 – Impact...,https://climatekg.tibwiki.io/entity/Q3326,A/R
7,https://climatekg.tibwiki.io/entity/Q150,Working Group III: Climate Change 2022 – Mitig...,https://climatekg.tibwiki.io/entity/Q3326,A/R
8,https://climatekg.tibwiki.io/entity/Q77,Working Group I: Climate Change 2021 – The Phy...,https://climatekg.tibwiki.io/entity/Q3039,A1B
9,https://climatekg.tibwiki.io/entity/Q57,Special Report: The Ocean and Cryosphere in a ...,https://climatekg.tibwiki.io/entity/Q2901,AABW


## Acronym questions

From the GitHub issue: https://github.com/TIBHannover/ClimateKG-Data-Bench/issues/1

 1. How many acronyms does each of the AR6 reports define?

## Question 1

Which acronyms appear in all three working group reports?

This cell identifies acronyms linked to Working Group I, II, and III reports via `P3` and provides the reusable SPARQL query.

In [3]:
query_report_counts = f'''
PREFIX wd: <{entity_ns}>
PREFIX wdt: <{property_ns}>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?report ?reportLabel (COUNT(DISTINCT ?acronym) AS ?acronymCount)
WHERE {{
  ?acronym wdt:P1 wd:Q2087 ;
           wdt:P3 ?report .
  ?report wdt:P1 wd:Q4 ;
          rdfs:label ?reportLabel .
  FILTER(LANG(?reportLabel) = "en")
  FILTER(
    CONTAINS(?reportLabel, "Working Group") ||
    CONTAINS(?reportLabel, "Special Report") ||
    CONTAINS(?reportLabel, "Synthesis Report")
  )
}}
GROUP BY ?report ?reportLabel
ORDER BY ?reportLabel
'''

display(Markdown("**SPARQL query (copy/paste into ClimateKG Query UI: https://climatekg.tibwiki.io/query/)**"))
display(Markdown("```sparql\n" + query_report_counts + "\n```"))

sparql = build_sparql_client(SPARQL_ENDPOINT)
sparql.setQuery(query_report_counts)
sparql.setReturnFormat(JSON)
report_count_results = sparql.query().convert()
report_count_rows = report_count_results.get("results", {}).get("bindings", [])
report_counts_df = pd.DataFrame([{k: v["value"] for k, v in row.items()} for row in report_count_rows])

if report_counts_df.empty:
    print("No AR6 report-linked acronyms were found in the current graph.")
else:
    report_counts_df["reportQID"] = report_counts_df["report"].astype(str).str.rsplit("/", n=1).str[-1]
    report_counts_df = report_counts_df.sort_values("reportLabel").reset_index(drop=True)
    print(f"AR6 reports returned: {len(report_counts_df)}")
    display(report_counts_df[["reportQID", "reportLabel", "acronymCount"]])

**SPARQL query (copy/paste into ClimateKG Query UI: https://climatekg.tibwiki.io/query/)**

```sparql

PREFIX wd: <https://climatekg.tibwiki.io/entity/>
PREFIX wdt: <https://climatekg.tibwiki.io/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?report ?reportLabel (COUNT(DISTINCT ?acronym) AS ?acronymCount)
WHERE {
  ?acronym wdt:P1 wd:Q2087 ;
           wdt:P3 ?report .
  ?report wdt:P1 wd:Q4 ;
          rdfs:label ?reportLabel .
  FILTER(LANG(?reportLabel) = "en")
  FILTER(
    CONTAINS(?reportLabel, "Working Group") ||
    CONTAINS(?reportLabel, "Special Report") ||
    CONTAINS(?reportLabel, "Synthesis Report")
  )
}
GROUP BY ?report ?reportLabel
ORDER BY ?reportLabel

```

AR6 reports returned: 7


,reportQID,reportLabel,acronymCount
0,Q189,Climate Change 2023: Synthesis Report. Contrib...,89
1,Q35,Special Report: Climate Change and Land,425
2,Q10,Special Report: Global Warming of 1.5°C,561
3,Q57,Special Report: The Ocean and Cryosphere in a ...,193
4,Q77,Working Group I: Climate Change 2021 – The Phy...,516
5,Q106,Working Group II: Climate Change 2022 – Impact...,670
6,Q150,Working Group III: Climate Change 2022 – Mitig...,539


### How this answer was computed

1. The query selects acronym items using `P1 -> Q2087` (Acronym class).
2. It links each acronym to reports through `P3`.
3. It limits results to AR6 report labels (Working Group, Special Report, Synthesis Report).
4. It groups by report and counts distinct acronyms per report.
5. The result table shows one row per AR6 report with the total acronym count.

Because the full SPARQL query is shown above and run live against ClimateKG, the result is reproducible and can be validated in the query UI.

## Interpretation

If this notebook returns zero rows, that is also a valid result: it means the current live ClimateKG instance does not yet expose acronym or glossary entries under the expected class labels.

This is the important query-first check. Once the graph contains acronym or glossary items, the same pattern can be extended to count entries per report, summarise term families, and compare label coverage across sections and chapters.